# ASHRAE Great Energy Predictor III — IoT Data Cleaning, Feature Engineering & SQL Analytics

**Course:** ISTM 6217 - Internet of Things Management
**Dataset:** [ASHRAE - Great Energy Predictor III (Kaggle)](https://www.kaggle.com/competitions/ashrae-energy-prediction)
**Group Members:** Surname Name

<a id="toc"></a>
## Table of Contents
1. [Environment Setup & Library Imports](#1-env-setup)
2. [Dataset Download & Extraction](#2-data-download)
3. [Spark Session & Data Loading](#3-spark-loading)
4. [Data Cleaning & Feature Engineering](#4-cleaning-fe)
   * [4.1 Building Metadata Cleaning & Age Cohorts](#4-1-building-metadata)
   * [4.2 Train Data Inspection](#4-2-train-data)
   * [4.3 Weather Data Cleaning & Time/Rolling Features](#4-3-weather-data)
   * [4.4 Building Size Quartiles & Schema Verification](#4-4-building-features)
5. [Business Questions & Spark SQL Analysis](#5-business-questions)
   * [5.1 Daily Operational Cycles](#5-1-daily-cycles)
   * [5.2 Weekly Usage Drops](#5-2-weekly-drops)
   * [5.3 Seasonal Fuel Switching](#5-3-seasonal-switching)
   * [5.4 Temperature Thresholds](#5-4-temp-thresholds)
   * [5.5 Weather & Building Age Efficiency](#5-5-age-efficiency)

<a id="1-env-setup"></a>
## 1. Environment Setup & Library Imports

In [ ]:
# Ignore PyArrow timezone warnings
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

In [ ]:
# Import Spark, pandas, plotting, and utility libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, isnull
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, TimestampType
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import Window
from pyspark.sql import functions as F
from pandas.tseries.holiday import USFederalHolidayCalendar
from pyspark.pandas.utils import PandasAPIOnSparkAdviceWarning
import pyspark.pandas as ps
import numpy as np
import zipfile
import warnings

# Ignore FutureWarnings generated by the PySpark Pandas API
warnings.simplefilter(action='ignore', category=FutureWarning)
from pyspark.pandas.utils import PandasAPIOnSparkAdviceWarning

# Prevent pandas-on-spark from failing on ANSI mode and avoid heavy sequence indexing
ps.set_option("compute.fail_on_ansi_mode", False)
ps.set_option("compute.default_index_type", "distributed")

<a id="2-data-download"></a>
## 2. Dataset Download & Extraction
Once your [Kaggle API credentials](https://www.kaggle.com/docs/api) are configured, run the cell below to download and extract the [ASHRAE competition dataset](https://www.kaggle.com/competitions/ashrae-energy-prediction/data).

### Configure Kaggle Credentials via Colab Secrets

In [ ]:
import os
import json
from google.colab import userdata

# Retrieve your API token secrets securely from Colab Secrets
kaggle_user = userdata.get('KAGGLE_USERNAME').strip()
kaggle_key = userdata.get('KAGGLE_KEY').strip()

# Setup directory and write kaggle.json
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

token_path = os.path.join(kaggle_dir, 'kaggle.json')
with open(token_path, 'w') as f:
    json.dump({'username': kaggle_user, 'key': kaggle_key}, f)

os.chmod(token_path, 0o600)
print(f"Successfully configured kaggle.json for user: {kaggle_user}!")

In [ ]:
competition_name = 'ashrae-energy-prediction'
zip_path = f"{competition_name}.zip"

# Define the path to the data folder
extract_dir = 'data/'

if not os.path.exists(extract_dir) or len(os.listdir(extract_dir)) == 0:
    print("Downloading dataset...")
    !kaggle competitions download -c {competition_name}

    if os.path.exists(zip_path):
        print(f"Extracting files into {extract_dir}/...")
        os.makedirs(extract_dir, exist_ok=True)
        import zipfile
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        print("Files extracted successfully!")
    else:
        print("Download failed. Verify that your competition credentials have accepted the rules on the Kaggle competition page.")
else:
    print(f"Data folder '{extract_dir}' already exists and contains files.")

<a id="3-spark-loading"></a>
## 3. Spark Session & Data Loading
Initialize the `SparkSession`, load the raw CSV files into Spark DataFrames, and register temporary SQL views (`train`, `weather_train`, `building_metadata`).

In [ ]:
spark = SparkSession.builder \
    .appName("ASHRAE Energy Prediction") \
    .getOrCreate()

In [ ]:
# Load the datasets into Spark DataFrames with headers and inferred schemas
building_metadata_df = spark.read.csv(f"{extract_dir}building_metadata.csv", header=True, inferSchema=True)
train_df = spark.read.csv(f"{extract_dir}train.csv", header=True, inferSchema=True)
weather_train_df = spark.read.csv(f"{extract_dir}weather_train.csv", header=True, inferSchema=True)

# Register DataFrames as temporary SQL views
train_df.createOrReplaceTempView("train")
weather_train_df.createOrReplaceTempView("weather_train")
building_metadata_df.createOrReplaceTempView("building_metadata")

In [ ]:
# Print the schema for each DataFrame
building_metadata_df.printSchema()
train_df.printSchema()
weather_train_df.printSchema()

In [ ]:
# Disable ANSI mode for pandas-on-Spark compatibility
spark.conf.set("spark.sql.ansi.enabled", "false")

<a id="4-cleaning-fe"></a>
## 4. Data Cleaning & Feature Engineering
* **Time features:** `hour_of_day`, `day_of_week` (Mon=0), `month`, `is_weekend`, `is_holiday` (US federal holidays), and `us_season`
* **Weather features:** `temp_threshold` and `air_temp_avg_24h` – rolling average air temperature over the last 24 hours, per site
* **Building features:** `building_age_group` (Historic / Older / Modern / New) and `building_size` (small / medium / large / massive by square-feet quartile)

<a id="4-1-building-metadata"></a>
### 4.1 Building Metadata Cleaning & Age Cohorts
[↑ Back to Table of Contents](#toc)

In [ ]:
# Query all records from the building_metadata table
building_metadata = spark.sql("""
    SELECT *
    FROM building_metadata
""")

In [ ]:
# Convert building_metadata to a pandas-on-Spark DataFrame
building_metadata = building_metadata_df.pandas_api()

In [ ]:
# Check for missing values per column
building_metadata.isnull().sum()

In [ ]:
# Fill floor_count with a logical default of 1
building_metadata['floor_count'] = building_metadata['floor_count'].fillna(1)

# Fill year_built with the median year
median_year = building_metadata['year_built'].median()
building_metadata['year_built'] = building_metadata['year_built'].fillna(median_year)

In [ ]:
# Verify no missing values remain in building_metadata
building_metadata.isnull().sum()

In [ ]:
# 1. Bring building_metadata to local pandas
bm = building_metadata.to_pandas()

# 2. Define the bins and labels for the 4 categories
age_bins = [-np.inf, 1959, 1979, 1999, np.inf]
age_labels = [
    'Historic', # < 1960
    'Older',    # 1960 - 1979
    'Modern',   # 1980 - 1999
    'New'       # 2000+
]

# 3. Create the feature
bm["building_age_group"] = pd.cut(
    bm["year_built"],
    bins=age_bins,
    labels=age_labels
).astype(str)

# 4. Push back to Spark and register the updated view
building_metadata = ps.from_pandas(bm)
building_metadata.to_spark(index_col="index").createOrReplaceTempView("building_metadata")

In [ ]:
# Register updated building_metadata temporary view
building_metadata.to_spark().createOrReplaceTempView("building_metadata")

<a id="4-2-train-data"></a>
### 4.2 Train Data Inspection
[↑ Back to Table of Contents](#toc)

In [ ]:
# Preview records from the train table
train = spark.sql("""
    SELECT *
    FROM train
""")

In [ ]:
# Convert train to a pandas-on-Spark DataFrame
train = train.pandas_api()

In [ ]:
# Register updated temporary views in Spark SQL
train.to_spark().createOrReplaceTempView("train")

<a id="4-3-weather-data"></a>
### 4.3 Weather Data Cleaning & Time/Rolling Features
[↑ Back to Table of Contents](#toc)

In [ ]:
# Preview the weather_train table
weather = spark.sql("""
    SELECT *
    FROM weather_train
""")

In [ ]:
# Convert weather to a pandas-on-Spark DataFrame
weather = weather.pandas_api()

In [ ]:
# Check for missing values in the weather table
weather.isnull().sum()

In [ ]:
# Ensure data is sorted chronologically before forward-filling
weather = weather.sort_values(by=['site_id', 'timestamp'])

In [ ]:
# Convert 7MB weather table to pandas for ultra-fast, zero-overhead cleaning
w_df = weather.to_pandas()
cols_ffill = ["air_temperature", "dew_temperature", "wind_speed", "wind_direction", "sea_level_pressure"]

# 1. Group by site_id and forward-fill, then backward-fill
w_df[cols_ffill] = w_df.groupby("site_id")[cols_ffill].ffill()
w_df[cols_ffill] = w_df.groupby("site_id")[cols_ffill].bfill()

# 2. Fill cloud coverage with median
w_df["cloud_coverage"] = w_df["cloud_coverage"].fillna(w_df["cloud_coverage"].median())

# 3. Fill missing precipitation values with 0
w_df["precip_depth_1_hr"] = w_df["precip_depth_1_hr"].fillna(0)

# 4. Fill remaining sea level pressure with global median
w_df["sea_level_pressure"] = w_df["sea_level_pressure"].fillna(w_df["sea_level_pressure"].median())

# Convert back to pandas-on-spark
weather = ps.from_pandas(w_df)

In [ ]:
# Verify no missing values remain in the weather table
weather.isnull().sum()

In [ ]:
# Get US federal holidays from 2015 to 2019
holidays = (USFederalHolidayCalendar()
            .holidays(start="2015-01-01", end="2019-12-31")
            .strftime("%Y-%m-%d")
            .tolist())

# Map month numbers to US seasons
season_mapping = {
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Fall', 10: 'Fall', 11: 'Fall'
}

# Define a 24-hour rolling window (current hour + previous 23 hours) per site
w_24h = (Window.partitionBy("site_id")
         .orderBy(F.col("timestamp").cast("long"))
         .rangeBetween(-23 * 3600, 0))


# Function to extract time, weekend, holiday, and season features
def add_time_features(df):
    df["hour_of_day"] = df["timestamp"].dt.hour
    df["day_of_week"] = df["timestamp"].dt.dayofweek
    df["month"] = df["timestamp"].dt.month
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
    df["is_holiday"] = df["timestamp"].dt.strftime("%Y-%m-%d").isin(holidays).astype(int)
    df["us_season"] = df["month"].map(season_mapping)
    return df

# Function to categorize temperature thresholds and calculate 24-hour average temperature
def add_advanced_weather_features(df):
    return (df.to_spark()
              .withColumn("temp_threshold",
                  F.when(F.col("air_temperature") < 0, 'Freezing')
                  .when((F.col("air_temperature") >= 0) & (F.col("air_temperature") <= 10), 'Cold')
                  .when((F.col("air_temperature") > 10) & (F.col("air_temperature") <= 20), 'Comfort')
                  .when((F.col("air_temperature") > 20) & (F.col("air_temperature") <= 30), 'Warm')
                  .otherwise('Extreme Heat')
              )
              .withColumn("air_temp_avg_24h", F.avg("air_temperature").over(w_24h)))


# Apply time and weather feature functions
weather = add_time_features(weather)
weather = add_advanced_weather_features(weather)

# Register updated weather view and convert back to pandas-on-Spark
weather.createOrReplaceTempView("weather")
weather = weather.pandas_api()

<a id="4-4-building-features"></a>
### 4.4 Building Size Quartiles & Schema Verification
[↑ Back to Table of Contents](#toc)

In [ ]:
# Categorize buildings into size quartiles
bm = building_metadata.to_pandas()
bm["building_size"] = pd.qcut(bm["square_feet"], q=4,
                              labels=["small", "medium", "large", "massive"]).astype(str)
building_metadata = ps.from_pandas(bm)

# 2. Add index_col="index" to resolve the missing index warning
building_metadata.to_spark(index_col="index").createOrReplaceTempView("building_metadata")

In [ ]:
# Print updated schemas to verify new features
building_metadata.to_spark().printSchema()
train.to_spark().printSchema()
weather.to_spark().printSchema()

<a id="5-business-questions"></a>
## 5. Business Questions & Spark SQL Analysis
[↑ Back to Table of Contents](#toc)

<a id="5-1-daily-cycles"></a>
### 5.1 Daily Operational Cycles
To analyze daily usage patterns, this query extracts the hour from `timestamp` and calculates the average `meter_reading`:

In [ ]:
# Calculate average meter reading for each hour of the day
result = spark.sql("""
SELECT
    EXTRACT(HOUR FROM timestamp) AS hour_of_day,
    AVG(meter_reading) AS avg_hourly_usage
FROM
    train
GROUP BY
    EXTRACT(HOUR FROM timestamp)
ORDER BY
    hour_of_day;
""")

# Display the hourly usage results
result.show()

In [ ]:
# Convert SQL result to pandas DataFrame for plotting
df_hourly = result.toPandas()

# Plot a line chart of average hourly energy usage
plt.figure(figsize=(10, 5))
plt.plot(df_hourly["hour_of_day"], df_hourly["avg_hourly_usage"], marker="o", linewidth=2, color="steelblue")
plt.title("Daily Operational Cycles: Average Hourly Energy Usage")
plt.xlabel("Hour of Day (0 - 23)")
plt.ylabel("Average Meter Reading")
plt.xticks(range(0, 24))
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

<a id="5-2-weekly-drops"></a>
### 5.2 Weekly Usage Drops
To compare weekday versus weekend usage, this query extracts the day of the week from `timestamp` to aggregate `meter_reading`:

In [ ]:
# Calculate average meter reading for each day of the week
result = spark.sql("""
SELECT
    EXTRACT(DAYOFWEEK FROM timestamp) AS day_of_week,
    AVG(meter_reading) AS avg_daily_usage
FROM
    train
GROUP BY
    EXTRACT(DAYOFWEEK FROM timestamp)
ORDER BY
    day_of_week;
""")

# Display the daily usage results
result.show()

In [ ]:
# Convert SQL result to pandas DataFrame and map day numbers to names
df_weekly = result.toPandas()
day_labels = {1: "Sun", 2: "Mon", 3: "Tue", 4: "Wed", 5: "Thu", 6: "Fri", 7: "Sat"}
df_weekly["day_name"] = df_weekly["day_of_week"].map(day_labels)

# Plot a bar chart of average energy usage by day of the week
plt.figure(figsize=(8, 5))
plt.bar(df_weekly["day_name"], df_weekly["avg_daily_usage"], color="teal", edgecolor="black")
plt.title("Weekly Usage Drops: Average Energy Usage by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Average Meter Reading")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

<a id="5-3-seasonal-switching"></a>
### 5.3 Seasonal Fuel Switching
To observe how different utility types fluctuate across seasons, this query joins `train`, `building_metadata`, and `weather` to group total `meter_reading` by US season (`us_season`) and `meter` type:

In [ ]:
# Join train, building_metadata, and weather to calculate total seasonal usage by US season and meter type
result = spark.sql("""
SELECT /*+ BROADCAST(b), BROADCAST(w) */
    w.us_season,
    t.meter,
    ROUND(SUM(t.meter_reading), 2) AS total_seasonal_usage
FROM
    train t
JOIN
    building_metadata b ON t.building_id = b.building_id
JOIN
    weather w ON b.site_id = w.site_id AND t.timestamp = w.timestamp
GROUP BY
    w.us_season,
    t.meter
ORDER BY
    CASE w.us_season
        WHEN 'Winter' THEN 1
        WHEN 'Spring' THEN 2
        WHEN 'Summer' THEN 3
        WHEN 'Fall' THEN 4
    END,
    t.meter;
""")

# Display the seasonal fuel switching results
result.show(20, truncate=False)

In [ ]:
# Convert SQL result to pandas DataFrame and map meter IDs to names
df_seasonal = result.toPandas()
meter_names = {0: "Electricity", 1: "Chilled Water", 2: "Steam", 3: "Hot Water"}
df_seasonal["meter_name"] = df_seasonal["meter"].map(meter_names)

# Order seasons logically: Winter -> Spring -> Summer -> Fall
season_order = ["Winter", "Spring", "Summer", "Fall"]
df_seasonal_pivot = df_seasonal.pivot(index="us_season", columns="meter_name", values="total_seasonal_usage").reindex(season_order)

# Plot grouped bar chart of seasonal utility consumption
df_seasonal_pivot.plot(kind="bar", figsize=(10, 5), edgecolor="black")
plt.title("Seasonal Fuel Switching: Total Energy Usage by US Season and Meter Type")
plt.xlabel("US Season")
plt.ylabel("Total Usage (Log Scale)")
plt.yscale("log")
plt.xticks(rotation=0)
plt.legend(title="Meter Type")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

<a id="5-4-temp-thresholds"></a>
### 5.4 Temperature Thresholds
To identify how energy consumption responds when outdoor temperatures cross critical heating and cooling thresholds, this query joins `train`, `building_metadata`, and `weather` to aggregate usage across temperature bands and meter types:

In [ ]:
# Join train, building_metadata, and weather to calculate average usage by temperature threshold and meter
result = spark.sql("""
SELECT /*+ BROADCAST(b), BROADCAST(w) */
    w.temp_threshold,
    t.meter,
    COUNT(t.meter_reading) AS reading_count,
    ROUND(AVG(t.meter_reading), 2) AS avg_meter_reading
FROM
    train t
JOIN
    building_metadata b ON t.building_id = b.building_id
JOIN
    weather w ON b.site_id = w.site_id AND t.timestamp = w.timestamp
GROUP BY
    w.temp_threshold,
    t.meter
ORDER BY
    w.temp_threshold,
    t.meter;
""")

# Display the temperature threshold results
result.show(20, truncate=False)

In [ ]:
# Convert SQL result to pandas DataFrame and map meter names
df_temp = result.toPandas()
meter_names = {0: "Electricity", 1: "Chilled Water", 2: "Steam", 3: "Hot Water"}
df_temp["meter_name"] = df_temp["meter"].map(meter_names)

# Order temperature thresholds logically from Freezing to Extreme Heat
threshold_order = ["Freezing", "Cold", "Comfort", "Warm", "Extreme Heat"]
df_temp_pivot = df_temp.pivot(index="temp_threshold", columns="meter_name", values="avg_meter_reading").reindex(threshold_order)

# Plot grouped bar chart of average usage across temperature thresholds
df_temp_pivot.plot(kind="bar", figsize=(10, 5), edgecolor="black")
plt.title("Temperature Thresholds: Average Meter Reading by Temperature Band")
plt.xlabel("Outdoor Temperature Threshold")
plt.ylabel("Average Meter Reading (Log Scale)")
plt.yscale("log")
plt.xticks(rotation=0)
plt.legend(title="Meter Type")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

<a id="5-5-age-efficiency"></a>
### 5.5 Weather & Building Age Efficiency
To evaluate how building age impacts energy efficiency (usage per square foot) under varying weather conditions and seasons, this query groups buildings by construction era (`building_age_group`) and season (`us_season`).

[↑ Back to Table of Contents](#toc)

In [ ]:
# Join tables to compare average usage and energy per square foot by building age group and season
result = spark.sql("""
SELECT /*+ BROADCAST(b), BROADCAST(w) */
    b.building_age_group,
    w.us_season,
    ROUND(AVG(w.air_temp_avg_24h), 2) AS avg_24h_temp,
    ROUND(AVG(t.meter_reading), 2) AS avg_meter_reading,
    ROUND(AVG(t.meter_reading / b.square_feet), 4) AS avg_kwh_per_sqft
FROM
    train t
JOIN
    building_metadata b ON t.building_id = b.building_id
JOIN
    weather w ON b.site_id = w.site_id AND t.timestamp = w.timestamp
GROUP BY
    b.building_age_group,
    w.us_season
ORDER BY
    b.building_age_group,
    w.us_season;
""")

In [ ]:
# Display the building age efficiency results
result.show(20, truncate=False)

# Convert SQL result to pandas DataFrame for plotting
df_age = result.toPandas()
age_order = ["Historic", "Older", "Modern", "New"]
season_order = ["Winter", "Spring", "Summer", "Fall"]

# Pivot data to compare energy intensity per sqft across building age groups and seasons
df_age_pivot = df_age.pivot(index="building_age_group", columns="us_season", values="avg_kwh_per_sqft").reindex(index=age_order, columns=season_order)

# Plot grouped bar chart of energy efficiency by building age group and season
df_age_pivot.plot(kind="bar", figsize=(10, 5), edgecolor="black")
plt.title("Weather & Building Age Efficiency: Average Energy per Sqft by Era and Season")
plt.xlabel("Building Age Group")
plt.ylabel("Average Usage per Square Foot (kWh / sqft)")
plt.xticks(rotation=0)
plt.legend(title="US Season")
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()